# Pipeline layer review

Read-only look at what the jobs wrote, using **pandas** rather than Spark -- no
JVM, no session startup, no Iceberg jars. Loads in a second or two.

The path logic lives in `review_layers.py` beside this notebook, which derives
every path from `config/local.yaml` through the pipeline's own `build_path`. So
this notebook cannot drift from where the jobs actually write.

**Read-only by design.** Nothing here writes, deletes or moves an S3 object.

In [ ]:
import sys
from datetime import datetime

import pandas as pd

sys.path.insert(0, "..")   # the python_scripts package, from the repo root
sys.path.insert(0, ".")    # review_layers.py, beside this notebook

import review_layers as rl

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

cfg, fs = rl.connect("local")
print(f"bucket: {cfg.bucket}   env: {cfg.env}   region: {cfg.region}")
print(f"tables: {sorted(cfg.tables)}")

## 1. What exists

Start here after any run. One row per prefix holding parquet, so an empty
result means the job wrote nothing -- and a suspiciously small `bytes` means it
wrote an empty frame: schema only, no rows.

In [ ]:
rl.list_prefixes(cfg, fs)

## 2. Was anything held back?

Reason counts across both stages and every table. This is the health check --
run it before trusting a batch.

`raw_ingest` can only reject a null business key. Cast failures need a target
type to try, so they can only surface in `raw_to_silver`.

In [ ]:
rl.reject_summary(cfg, fs)

## 3. The held rows themselves

With their **original** values, because the split runs before any conversion.
A transformed copy would be useless for working out what the source contained.

`batch_date=None` reads every batch at once; pass a date to narrow it.

In [ ]:
held = rl.read_quarantine(cfg, fs, "order_items", "raw_ingest")

print(f"{len(held)} row(s) held")
held

In [ ]:
# Just the columns that matter for triage
triage = ["ORDER_ID", "LINEITEM_ID", "ITEM_PRICE", "_reject_reason",
          "_quarantine_stage", "_quarantine_table", "_batch_date"]

held[triage] if not held.empty else "nothing held"

In [ ]:
# The other stage. Empty until raw_to_silver has run.
rl.read_quarantine(cfg, fs, "order_items", "raw_to_silver")

## 4. Raw layer

A faithful copy of the source, partitioned by **when rows arrived** (the batch
date) rather than when the orders were created.

In [ ]:
raw = rl.read_raw(cfg, fs, "order_items", datetime(2026, 10, 16))

print(f"{len(raw):,} rows")
raw.head()

In [ ]:
# date_dim is load_type: static, so it has no batch partition -- omit the date.
rl.read_raw(cfg, fs, "date_dim").head()

## 5. Checks worth running after a load

Cheap questions that catch the failure modes this pipeline actually has.

In [ ]:
# Does the watermark line up with the --period you passed? Every row of an
# initial load carries the single instant the ALTER TABLE ran.
raw["inserted_at"].agg(["min", "max"])

In [ ]:
# Will CREATION_TIME_UTC survive the silver cast? A non-ISO value gets
# quarantined, so an unexpected format here means an empty silver table.
parsed = pd.to_datetime(raw["CREATION_TIME_UTC"], format="ISO8601",
                        errors="coerce", utc=True)

print(f"unparseable: {parsed.isna().sum()} of {len(parsed):,}")
print(f"event-time range: {parsed.min()}  ->  {parsed.max()}")

In [ ]:
# Are the merge keys unique? Duplicates are quarantined rather than merged, so
# a surprise here means rows you expected in silver are being held instead.
keys = cfg.get_table_config("order_items").business_keys
group_sizes = raw.groupby(keys).size()

print(f"duplicate key groups: {(group_sizes > 1).sum()}")
group_sizes[group_sizes > 1].head()

In [ ]:
# Null business keys -- these are what raw_ingest quarantines.
raw[keys].isna().sum()

## A note on the silver layer

Silver is **not** in S3 when you run `--env local`. `create_spark_session`
configures the Iceberg catalog with `warehouse = file:///<repo>/warehouse`, so
silver tables land on local disk while raw goes to S3.

That is also why pandas cannot read silver the way it reads the layers above.
An Iceberg table is a set of parquet files *plus metadata deciding which of
them are live*; reading the parquet directly would show rows that later
snapshots deleted.

To query silver or gold, use Spark -- see **`review_gold.ipynb`** beside this
notebook, which does exactly that:

```python
spark.table("iceberg.silver.order_items")
spark.table("iceberg.gold.daily_restaurant_sales")
```

The catalog is named `iceberg` and serves every Iceberg layer; the namespace
is the layer name from `layers:` in config. `pyiceberg` pointed at the
warehouse would also work and starts far faster, at the cost of another
dependency.